# Page Replacement Algorithm Simulator
**Course:** Operating Systems | **Topic:** Virtual Memory / Page Replacement

**Name:** Shashwat Shandilya  |  **Roll No.:** BTE25DTA100011

Implements FIFO, LRU and Optimal page replacement with step-by-step traces and a final comparison.

## 1. Algorithms
Each function returns `(faults, hits, trace)`. A trace row is `(step, page, frames, status, replaced)`. All frames start empty (`None`).

In [17]:
def _fmt(frames):
    return [p for p in frames]

def fifo(reference, num_frames):
    frames = [None]*num_frames
    ptr = 0                      # next slot to replace (oldest page)
    faults = hits = 0
    trace = []
    for step, page in enumerate(reference, 1):
        replaced = None
        if page in frames:
            hits += 1; status = "Hit"
        else:
            faults += 1; status = "Fault"
            if None in frames:
                frames[frames.index(None)] = page
            else:
                replaced = frames[ptr]
                frames[ptr] = page
                ptr = (ptr + 1) % num_frames
        trace.append((step, page, list(frames), status, replaced))
    return faults, hits, trace

def lru(reference, num_frames):
    frames = [None]*num_frames
    last_used = {}               # page -> last time it was referenced
    faults = hits = 0
    trace = []
    for step, page in enumerate(reference, 1):
        replaced = None
        if page in frames:
            hits += 1; status = "Hit"
        else:
            faults += 1; status = "Fault"
            if None in frames:
                frames[frames.index(None)] = page
            else:
                victim = min(frames, key=lambda p: last_used[p])
                replaced = victim
                frames[frames.index(victim)] = page
        last_used[page] = step
        trace.append((step, page, list(frames), status, replaced))
    return faults, hits, trace

def optimal(reference, num_frames):
    frames = [None]*num_frames
    faults = hits = 0
    trace = []
    n = len(reference)
    for i, page in enumerate(reference):
        step = i + 1
        replaced = None
        if page in frames:
            hits += 1; status = "Hit"
        else:
            faults += 1; status = "Fault"
            if None in frames:
                frames[frames.index(None)] = page
            else:
                best_slot, best_dist = 0, -1
                for slot, p in enumerate(frames):        # slots scanned in order -> ties pick first
                    if p in reference[i+1:]:
                        dist = reference[i+1:].index(p)
                    else:
                        dist = n + 1                      # never used again
                    if dist > best_dist:
                        best_slot, best_dist = slot, dist
                replaced = frames[best_slot]
                frames[best_slot] = page
        trace.append((step, page, list(frames), status, replaced))
    return faults, hits, trace

## 2. Input and validation

In [18]:
def read_input():
    """Read reference string and frame count from the user; reject invalid input."""
    raw = input("Enter page reference string (comma or space separated): ").strip()
    tokens = raw.replace(",", " ").split()
    if not tokens:
        raise ValueError("Reference string is empty.")
    reference = []
    for t in tokens:
        if not t.lstrip("-").isdigit():
            raise ValueError(f"Invalid page '{t}': pages must be integers.")
        if int(t) < 0:
            raise ValueError(f"Invalid page '{t}': pages cannot be negative.")
        reference.append(int(t))

    f = input("Enter number of frames: ").strip()
    if not f.lstrip("-").isdigit():
        raise ValueError(f"Invalid frame count '{f}': must be an integer.")
    num_frames = int(f)
    if num_frames < 1:
        raise ValueError("Number of frames must be at least 1.")
    return reference, num_frames

## 3. Output helpers

In [19]:
def print_trace(name, reference, num_frames, result):
    faults, hits, trace = result
    total = len(reference)
    print(f"\n{'=' * 64}\n{name}  (frames = {num_frames})\n{'=' * 64}")
    header = f"{'Step':<6}{'Page':<6}" + "".join(f"{'F' + str(i+1):<5}" for i in range(num_frames)) + f"{'Result':<8}{'Replaced':<9}"
    print(header)
    print("-" * len(header))
    for step, page, frames, status, replaced in trace:
        cells_ = "".join(f"{('-' if p is None else p)!s:<5}" for p in frames)
        rep = "-" if replaced is None else replaced
        print(f"{step:<6}{page:<6}{cells_}{status:<8}{rep!s:<9}")
    print(f"\nTotal Faults: {faults} | Total Hits: {hits} | "
          f"Hit Ratio: {hits/total*100:.2f}% | Fault Ratio: {faults/total*100:.2f}%")

def print_comparison(reference, results):
    total = len(reference)
    print(f"\n{'=' * 64}\nCOMPARISON\n{'=' * 64}")
    print(f"{'Algorithm':<12}{'Faults':<9}{'Hits':<7}{'Hit Ratio':<12}{'Fault Ratio':<12}")
    print("-" * 52)
    for name, (faults, hits, _) in results.items():
        print(f"{name:<12}{faults:<9}{hits:<7}{hits/total*100:>6.2f}%     {faults/total*100:>6.2f}%")

## 4. Run the simulator (enter input when prompted)

In [20]:
try:
    reference, num_frames = read_input()
    results = {
        "FIFO": fifo(reference, num_frames),
        "LRU": lru(reference, num_frames),
        "Optimal": optimal(reference, num_frames),
    }
    for name, res in results.items():
        print_trace(name, reference, num_frames, res)
    print_comparison(reference, results)
except ValueError as e:
    print("Error:", e)

Enter page reference string (comma or space separated): 1 2 3 1 4 2 5
Enter number of frames: 3

FIFO  (frames = 3)
Step  Page  F1   F2   F3   Result  Replaced 
--------------------------------------------
1     1     1    -    -    Fault   -        
2     2     1    2    -    Fault   -        
3     3     1    2    3    Fault   -        
4     1     1    2    3    Hit     -        
5     4     4    2    3    Fault   1        
6     2     4    2    3    Hit     -        
7     5     4    5    3    Fault   2        

Total Faults: 5 | Total Hits: 2 | Hit Ratio: 28.57% | Fault Ratio: 71.43%

LRU  (frames = 3)
Step  Page  F1   F2   F3   Result  Replaced 
--------------------------------------------
1     1     1    -    -    Fault   -        
2     2     1    2    -    Fault   -        
3     3     1    2    3    Fault   -        
4     1     1    2    3    Hit     -        
5     4     1    4    3    Fault   2        
6     2     1    4    2    Fault   3        
7     5     5    4    2  

## 5. Test cells (Part B and Part C experiments)

In [21]:
def run(ref, frames, show_trace=False):
    results = {"FIFO": fifo(ref, frames), "LRU": lru(ref, frames), "Optimal": optimal(ref, frames)}
    if show_trace:
        for n, r in results.items():
            print_trace(n, ref, frames, r)
    print_comparison(ref, results)
    return results

print("Sample run: 1 2 3 1 4 2 5, frames = 3")
run([1,2,3,1,4,2,5], 3)

print("\nPart B: 7,0,1,2,0,3,0,4,2,3,0,3,2, frames = 3")
run([7,0,1,2,0,3,0,4,2,3,0,3,2], 3, show_trace=True);

Sample run: 1 2 3 1 4 2 5, frames = 3

COMPARISON
Algorithm   Faults   Hits   Hit Ratio   Fault Ratio 
----------------------------------------------------
FIFO        5        2       28.57%      71.43%
LRU         6        1       14.29%      85.71%
Optimal     5        2       28.57%      71.43%

Part B: 7,0,1,2,0,3,0,4,2,3,0,3,2, frames = 3

FIFO  (frames = 3)
Step  Page  F1   F2   F3   Result  Replaced 
--------------------------------------------
1     7     7    -    -    Fault   -        
2     0     7    0    -    Fault   -        
3     1     7    0    1    Fault   -        
4     2     2    0    1    Fault   7        
5     0     2    0    1    Hit     -        
6     3     2    3    1    Fault   0        
7     0     2    3    0    Fault   1        
8     4     4    3    0    Fault   2        
9     2     4    2    0    Fault   3        
10    3     4    2    3    Fault   0        
11    0     0    2    3    Fault   4        
12    3     0    2    3    Hit     -        
13 

In [22]:
print("Experiment 1: Belady's anomaly")
belady = [1,2,3,4,1,2,5,1,2,3,4,5]
for f in (3, 4):
    print(f"\n--- {f} frames ---")
    run(belady, f)

Experiment 1: Belady's anomaly

--- 3 frames ---

COMPARISON
Algorithm   Faults   Hits   Hit Ratio   Fault Ratio 
----------------------------------------------------
FIFO        9        3       25.00%      75.00%
LRU         10       2       16.67%      83.33%
Optimal     7        5       41.67%      58.33%

--- 4 frames ---

COMPARISON
Algorithm   Faults   Hits   Hit Ratio   Fault Ratio 
----------------------------------------------------
FIFO        10       2       16.67%      83.33%
LRU         8        4       33.33%      66.67%
Optimal     6        6       50.00%      50.00%


In [23]:
print("Experiment 2: LRU better than FIFO")
run([1,3,5,1,3,4,1,3,3,3,5,1,2,1,3,1], 3)
print("\nExperiment 2: FIFO better than LRU")
run([5,2,4,4,5,2,3,2,2,4,3,1,4,5,1,2], 3);

Experiment 2: LRU better than FIFO

COMPARISON
Algorithm   Faults   Hits   Hit Ratio   Fault Ratio 
----------------------------------------------------
FIFO        10       6       37.50%      62.50%
LRU         7        9       56.25%      43.75%
Optimal     6        10      62.50%      37.50%

Experiment 2: FIFO better than LRU

COMPARISON
Algorithm   Faults   Hits   Hit Ratio   Fault Ratio 
----------------------------------------------------
FIFO        7        9       56.25%      43.75%
LRU         8        8       50.00%      50.00%
Optimal     6        10      62.50%      37.50%


In [24]:
print("Input validation demo (should show errors):")
for bad_ref, bad_frames in [("1 2 a 3", "3"), ("1 2 3", "0"), ("", "3")]:
    try:
        tokens = bad_ref.replace(",", " ").split()
        if not tokens: raise ValueError("Reference string is empty.")
        for t in tokens:
            if not t.isdigit(): raise ValueError(f"Invalid page '{t}': pages must be integers.")
        if int(bad_frames) < 1: raise ValueError("Number of frames must be at least 1.")
    except ValueError as e:
        print("Error:", e)

Input validation demo (should show errors):
Error: Invalid page 'a': pages must be integers.
Error: Number of frames must be at least 1.
Error: Reference string is empty.
